In [1]:
import glob
import json
import os
import sys
import zipfile
import requests

VERIFY_TLS = True   # corporate TLS-inspecting proxy? set False (disables cert checking)

def download_file(url, filename):
    response = requests.get(url, stream=True, verify=VERIFY_TLS)
    response.raise_for_status()
    with open(filename, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)

def make_json_array_file(folder, name):
    json_array = []
    for filename in sorted(glob.glob(os.path.join(folder, "*.json"))):
        with open(filename, "r", encoding="utf-8") as f:
            json_array.append(json.load(f))
    with open(name + ".json", "w", encoding="utf-8") as f:
        json.dump(json_array, f)
    print(f"{name}: {len(json_array)} records")

data = {
    "email": "your email account",
    "api_key": "your api_key found at account page in solar.chemdx.org",
}

ret = requests.post("https://solar.chemdx.org/api/v1/search", json=data, verify=VERIFY_TLS)
ret.raise_for_status()
ret_json = ret.json()

# Wrong credentials still return HTTP 200 with a status dict
if isinstance(ret_json, dict) and ret_json.get("status") == "failed":
    sys.exit(f"Authentication failed: {ret_json.get('message')}")

for g in ret_json:
    gid, gname = str(g["id"]), g["name"]
    zip_name = gid + ".zip"
    print(f"group {gid} ({gname}) ...")
    download_file(g["temporaryUrl"], zip_name)      # URL expires in ~5 min
    os.makedirs(gid, exist_ok=True)
    with zipfile.ZipFile(zip_name) as z:            # portable: no external 'unzip' needed
        z.extractall(gid)
    make_json_array_file(gid, gname)


In [2]:
import os
import zipfile

def extract_zip_files(folder_path):
    # Iterate over all files in the folder
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file.endswith(".zip"):
                file_path = os.path.join(root, file)
                # Extract the zip file
                with zipfile.ZipFile(file_path, 'r') as zip_ref:
                    zip_ref.extractall(root)
                # Optionally, delete the extracted zip file
                os.remove(file_path)

# Specify the folder path where the zip files are located
folder_path = "./"

# Call the function
extract_zip_files(folder_path)

In [4]:
import os
import json

def merge_json_files(folder_path, output_file):
    # Create an empty list to store the results
    merged_data = []

    # Iterate over each folder
    for folder_path in folder_paths:

        # Iterate over all files in the folder
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                if file.endswith(".json"):
                    file_path = os.path.join(root, file)

                    # Read the JSON file
                    with open(file_path, "r", encoding="utf-8") as f:
                        data = json.load(f)

                        # Extract only the information related to 'id' and 'input' to create new data
                        filtered_data = {
                            'id': data.get('id', None),
                            'author': data.get('author', None),
                            'input': data.get('input', None)                            
                        }
                        # This code only filteres out the JV data. 
                        # If you want to collect 'PL', 'absorption', 'SEM', 'XRD', and 'stability' data, please revise the code according to the JSON structure.
                        if 'analysis' in data and 'JV' in data['analysis']:
                            filtered_data['JV'] = data['analysis']['JV']                        
                        merged_data.append(filtered_data)

    # Save all data into a single JSON file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(merged_data, f, ensure_ascii=False, indent=4)

# Specify the folder paths and the output file name
folder_paths = ["./2","./3", "./6", "./7","./8", "./9", "./10", "./11", "./12", "./14"]
output_file = "./merge_json_recipes_JV.json"

# Call the function
merge_json_files(folder_paths, output_file)




In [ ]:
# Optional: keep only records that carry a specific measurement (e.g. SEM)
import glob, json, os, shutil

MEASUREMENT = "SEM"          # JV / UVVIS / XRD / PL / STABILITY / SEM / TRPL / GIWAXS ...
OUT = MEASUREMENT.lower() + "_only"

os.makedirs(OUT, exist_ok=True)
kept = 0
for path in glob.glob("*/*.json"):
    with open(path, encoding="utf-8") as f:
        rec = json.load(f)
    keys = set(rec.get("analysis") or {}) | set(rec.get("analysisInfo") or {})
    if any(MEASUREMENT.upper() in k.upper() for k in keys):
        dst = os.path.join(OUT, os.path.basename(os.path.dirname(path)))
        os.makedirs(dst, exist_ok=True)
        shutil.copy2(path, dst)
        kept += 1
print(f"{MEASUREMENT}: {kept} records -> {OUT}/")
